In [0]:
# Databricks notebook source
# =========================================================================
# STEP 1: ENVIRONMENT CONTROL AND LIVE-RELOAD (Developer Best Practice)
# =========================================================================
%load_ext autoreload
%autoreload 2

import sys
import os
import importlib

# Blokada generowania skompilowanego cache .pyc na klastrze chmurowym
sys.dont_write_bytecode = True

# Dynamically attach project root directory to Python paths
root_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if root_path not in sys.path:
    sys.path.insert(0, root_path)

# Force clean reload of modules from disk to cluster RAM
import src.readers.dataframe_reader
import src.rules.physical_rules
import src.rules.query_rules

importlib.reload(src.readers.dataframe_reader)
importlib.reload(src.rules.physical_rules)
importlib.reload(src.rules.query_rules)

# =========================================================================
# STEP 2: CORE ENGINE IMPORTS AND FULL PACKAGE OF 10 RULES
# =========================================================================
from apm_spark_auditor.auditor.engine import PerformanceEngine
from apm_spark_auditor.readers.dataframe_reader import DataFrameExplainReader
from apm_spark_auditor.policies.policy_manager import PolicyManager
from apm_spark_auditor.context.environment_provider import EnvironmentProvider
from apm_spark_auditor.suggestions.remediation_engine import RemediationEngine
from apm_spark_auditor.finops.cost_translator import CostTranslator
from apm_spark_auditor.reporters.console_reporter import ConsoleReporter

# Imports of physical and infrastructure rules
from apm_spark_auditor.rules.physical_rules import (
    SmallFilesRule,  # PERF-001
    MissedBroadcastRule,  # PERF-002
    OverPartitioningRule,  # PERF-004
    DataSkewRule,  # PERF-006
    MissingOptimizationRule,  # PERF-007
)

# Imports of logical rules and Catalyst optimizations
from apm_spark_auditor.rules.query_rules import (
    TypeCastingRule,  # PERF-003
    CartesianProductRule,  # PERF-005
    ExplodeAbuseRule,  # PERF-008
    RedundantScanRule,  # PERF-009
    NonVectorizedUdfRule,  # PERF-010
)


# =========================================================================
# STEP 3: UNIVERSAL ORCHESTRATING FUNCTION (Unified Engine)
# =========================================================================
def audit_production_process(
    context_name: str, table_name: str = None, df=None, source_volume_path: str = None
):
    """
    Runs full series of 10 diagnostic APM & FinOps tests
    for one specific data processing process.

    :param context_name: Unique name identifying the process (report label)
    :param table_name: Table/view name in Unity Catalog (optional - under Serverless)
    :param df: PySpark DataFrame instance examined in flight (optional)
    :param source_volume_path: Physical path to file directory/volume (optional)
    """
    print("\n" + "═" * 70)
    print(f"🎬 STARTING COMPLETE APM DIAGNOSTICS FOR: {context_name}")
    print("═" * 70, flush=True)

    # 1. Initialize technology stack (Dependency injection)
    policy_mgr = PolicyManager()
    env_prov = EnvironmentProvider(spark)
    remedy_eng = RemediationEngine()
    cost_trans = CostTranslator(policy_mgr.get_policy("finops"))
    reporter = ConsoleReporter()

    # 2. Definition and consolidation of complete set of 10 detectives
    complete_diagnostic_suite = [
        SmallFilesRule(),  # PERF-001
        MissedBroadcastRule(),  # PERF-002
        TypeCastingRule(),  # PERF-003
        OverPartitioningRule(),  # PERF-004
        CartesianProductRule(),  # PERF-005
        DataSkewRule(),  # PERF-006
        MissingOptimizationRule(),  # PERF-007
        ExplodeAbuseRule(),  # PERF-008
        RedundantScanRule(),  # PERF-009
        NonVectorizedUdfRule(),  # PERF-010
    ]

    # 3. Dynamic DataFrame resolution if only table name is provided
    resolved_df = df
    if resolved_df is None and table_name:
        try:
            resolved_df = spark.read.table(table_name)
        except Exception as e:
            print(f"⚠️  [PRE-FLIGHT-WARN] Cannot load table {table_name}: {str(e)}")

    # 4. Konstrukcja polimorficznego czytnika planu wykonania
    reader = DataFrameExplainReader(
        spark=spark, df=resolved_df, table_name=table_name, source_volume_path=source_volume_path
    )

    # 5. Assemble orchestrator and launch full audit loop
    engine = PerformanceEngine(
        reader=reader,
        rules=complete_diagnostic_suite,
        policy_manager=policy_mgr,
        env_provider=env_prov,
        remediation_engine=remedy_eng,
        cost_translator=cost_trans,
        reporter=reporter,
    )

    engine.run_audit(context_name=context_name)
    print(f"🏁 Completed diagnostic series for: {context_name}\n" + "═" * 70)
    audit_production_process(
        context_name="ETL-01-BESS-Telemetry",
        table_name="workspace.bronze.bess_telemetry_small_files",
        source_volume_path="/Volumes/workspace/default/weather_data/raw_source/bess_telemetry_raw",
    )


audit_production_process(
    context_name="ETL-03-Station-Enrichment",
    table_name="workspace.bronze.enriched_telemetry_heavy_shuffle",
)

In [0]:
from pyspark.sql import functions as F

# Create process under test in flight (e.g., PV pipeline with type casting)
df_pv_pipeline = spark.read.table("workspace.bronze.pv_metrics_string_nightmare").withColumn(
    "calculated_factor", F.explode(F.array(F.lit(1), F.lit(2)))
)

# Launch diagnostics providing clean business object
audit_production_process(context_name="ETL-02-PV-Pipeline-AdHoc", df=df_pv_pipeline)